# Đánh Giá Hiệu Năng

Notebook này đo thời gian thực thi 4 thuật toán đã triển khai bằng `timeit` và xuất kết quả dưới dạng bảng LaTeX.

Thiết kế benchmark:
- Mỗi thuật toán có 4 case theo kích thước đầu vào tăng dần
- Mỗi case đo tác vụ chính:
  - AES: Mã hóa + Giải mã
  - SHA256: Băm tin nhắn
  - EC Asymmetric: Mã hóa + Giải mã
  - Schnorr: Ký số + Xác minh

Các chỉ số báo cáo cho mỗi case:
- Min (ms)
- Max (ms)
- Avg (ms)



In [ ]:
import timeit
import statistics
from IPython.display import display, Latex

from aes import AES
from sha256 import SHA256
from ec_asymmetric import ECCAsymmetricEncryption
from schnorr_digital_signature import SchnorrDigitalSignature

In [ ]:
def benchmark_callable(func, number=10, repeat=7):
    """Trả về min/max/avg thời gian (ms) cho mỗi lần gọi hàm."""
    raw_times = timeit.repeat(func, number=number, repeat=repeat)
    per_call_ms = [(t / number) * 1000.0 for t in raw_times]
    min_ms = min(per_call_ms)
    max_ms = max(per_call_ms)
    avg_ms = statistics.mean(per_call_ms)
    return min_ms, max_ms, avg_ms


def latex_escape(text):
    return str(text).replace('_', '\\_')


def make_latex_table(rows):
    lines = []
    lines.append('\\begin{table}[h]')
    lines.append('\\centering')
    lines.append('\\begin{tabular}{l l l r r r}')
    lines.append('\\hline')
    lines.append('Thuật toán & Tác vụ & Kích thước dữ liệu & Min (ms) & Max (ms) & Avg (ms) \\\\')
    lines.append('\\hline')
    for row in rows:
        lines.append(
            f"{latex_escape(row['algorithm'])} & {latex_escape(row['operation'])} & {latex_escape(row['input_size'])} & {row['min_ms']:.4f} & {row['max_ms']:.4f} & {row['avg_ms']:.4f} \\\\"
        )
    lines.append('\\hline')
    lines.append('\\end{tabular}')
    lines.append('\\caption{Đánh giá hiệu năng theo 4 mức độ khó cho mỗi thuật toán (timeit).}')
    lines.append('\\label{tab:danh-gia-hieu-nang-4-muc-do}')
    lines.append('\\end{table}')
    return '\n'.join(lines)

In [ ]:
results = []

levels = [
    ('Dễ', 1),
    ('Trung bình', 2),
    ('Khá khó', 3),
    ('Khó', 4),
]

# 1) AES - Mã hóa + Giải mã (round-trip)
aes_key = 'LeHuynhAnhKhoi24'  # AES-192
aes_sizes = [64, 256, 1024, 4096]

for (level_name, _), size in zip(levels, aes_sizes):
    plaintext = ('A' * size)

    def aes_roundtrip():
        aes = AES(aes_key, mode='cbc')
        ct = aes.encrypt(plaintext)
        _ = aes.decrypt(ct)

    number = 5 if size <= 1024 else 2
    repeat = 7 if size <= 1024 else 5
    min_t, max_t, avg_t = benchmark_callable(aes_roundtrip, number=number, repeat=repeat)

    results.append({
        'algorithm': 'AES',
        'level': level_name,
        'operation': 'Mã hóa + Giải mã CBC',
        'input_size': f'Bản rõ ASCII {size} byte',
        'min_ms': min_t,
        'max_ms': max_t,
        'avg_ms': avg_t,
    })

# 2) SHA256 - Băm tin nhắn
sha_sizes = [64, 256, 1024, 4096]

for (level_name, _), size in zip(levels, sha_sizes):
    message = ('H' * size).encode('ascii')

    def sha_hash():
        _ = SHA256(message).generating_hash()

    number = 100 if size <= 1024 else 40
    repeat = 7
    min_t, max_t, avg_t = benchmark_callable(sha_hash, number=number, repeat=repeat)

    results.append({
        'algorithm': 'SHA256',
        'level': level_name,
        'operation': 'Băm tin nhắn',
        'input_size': f'Tin nhắn bytes {size} byte',
        'min_ms': min_t,
        'max_ms': max_t,
        'avg_ms': avg_t,
    })

# 3) EC Asymmetric - Mã hóa + Giải mã
ec_alice = ECCAsymmetricEncryption()
ec_bob = ECCAsymmetricEncryption()
_, ec_alice_pk = ec_alice.gen_key_pair()
_, ec_bob_pk = ec_bob.gen_key_pair()
ec_sizes = [8, 16, 32, 64]

for (level_name, _), size in zip(levels, ec_sizes):
    message = ('E' * size)

    def ec_roundtrip():
        payload = ec_alice.encrypt(message, ec_bob_pk)
        _ = ec_bob.decrypt(payload)

    min_t, max_t, avg_t = benchmark_callable(ec_roundtrip, number=1, repeat=5)

    results.append({
        'algorithm': 'EC_Asymmetric',
        'level': level_name,
        'operation': 'Mã hóa + Giải mã',
        'input_size': f'Tin nhắn ASCII {size} byte',
        'min_ms': min_t,
        'max_ms': max_t,
        'avg_ms': avg_t,
    })

# 4) Schnorr - Ký số + Xác minh
schnorr = SchnorrDigitalSignature()
p, q, a = schnorr._generate_parameters(p_bits=512, q_bits=160)
schnorr.p, schnorr.q, schnorr.a = p, q, a
schnorr_sk, schnorr_pk = schnorr.gen_key_pair()
schnorr_sizes = [32, 256, 1024, 4096]

for (level_name, _), size in zip(levels, schnorr_sizes):
    message = ('S' * size)

    def schnorr_roundtrip():
        sig = schnorr.sign(message, schnorr_sk)
        _ = schnorr.verify(message, sig, schnorr_pk)

    number = 10 if size <= 1024 else 5
    repeat = 7
    min_t, max_t, avg_t = benchmark_callable(schnorr_roundtrip, number=number, repeat=repeat)

    results.append({
        'algorithm': 'Schnorr',
        'level': level_name,
        'operation': 'Ký số + Xác minh',
        'input_size': f'Tin nhắn ASCII {size} byte',
        'min_ms': min_t,
        'max_ms': max_t,
        'avg_ms': avg_t,
    })

results

[{'algorithm': 'AES',
  'level': 'Dễ',
  'operation': 'Mã hóa + Giải mã CBC',
  'input_size': 'Bản rõ ASCII 64 byte',
  'min_ms': 6.491839996306226,
  'max_ms': 8.821059996262193,
  'avg_ms': 7.893445712813575},
 {'algorithm': 'AES',
  'level': 'Trung bình',
  'operation': 'Mã hóa + Giải mã CBC',
  'input_size': 'Bản rõ ASCII 256 byte',
  'min_ms': 22.14576000114903,
  'max_ms': 27.93964000302367,
  'avg_ms': 24.050282857414068},
 {'algorithm': 'AES',
  'level': 'Khá khó',
  'operation': 'Mã hóa + Giải mã CBC',
  'input_size': 'Bản rõ ASCII 1024 byte',
  'min_ms': 82.78775999788195,
  'max_ms': 95.92718000058085,
  'avg_ms': 89.70409714134543},
 {'algorithm': 'AES',
  'level': 'Khó',
  'operation': 'Mã hóa + Giải mã CBC',
  'input_size': 'Bản rõ ASCII 4096 byte',
  'min_ms': 324.48875000409316,
  'max_ms': 367.9178999882424,
  'avg_ms': 346.731399995042},
 {'algorithm': 'SHA256',
  'level': 'Dễ',
  'operation': 'Băm tin nhắn',
  'input_size': 'Tin nhắn bytes 64 byte',
  'min_ms': 0.580

In [ ]:
latex_table = make_latex_table(results)
print(latex_table)
display(Latex(latex_table))

\begin{table}[h]
\centering
\begin{tabular}{l l l r r r}
\hline
Thuật toán & Tác vụ & Kích thước dữ liệu & Min (ms) & Max (ms) & Avg (ms) \\
\hline
AES & Mã hóa + Giải mã CBC & Bản rõ ASCII 64 byte & 6.4918 & 8.8211 & 7.8934 \\
AES & Mã hóa + Giải mã CBC & Bản rõ ASCII 256 byte & 22.1458 & 27.9396 & 24.0503 \\
AES & Mã hóa + Giải mã CBC & Bản rõ ASCII 1024 byte & 82.7878 & 95.9272 & 89.7041 \\
AES & Mã hóa + Giải mã CBC & Bản rõ ASCII 4096 byte & 324.4888 & 367.9179 & 346.7314 \\
SHA256 & Băm tin nhắn & Tin nhắn bytes 64 byte & 0.5808 & 0.7002 & 0.6251 \\
SHA256 & Băm tin nhắn & Tin nhắn bytes 256 byte & 1.5206 & 1.6757 & 1.5812 \\
SHA256 & Băm tin nhắn & Tin nhắn bytes 1024 byte & 5.2398 & 6.1489 & 5.7892 \\
SHA256 & Băm tin nhắn & Tin nhắn bytes 4096 byte & 20.6710 & 24.6121 & 22.3833 \\
EC\_Asymmetric & Mã hóa + Giải mã & Tin nhắn ASCII 8 byte & 38.7697 & 45.7243 & 42.2926 \\
EC\_Asymmetric & Mã hóa + Giải mã & Tin nhắn ASCII 16 byte & 31.3071 & 53.0317 & 41.0478 \\
EC\_Asymmetric &

<IPython.core.display.Latex object>